# NYC Tripdata Lakehouse — Welcome Index

This repository implements a **Bronze → Silver → Gold** lakehouse flow for NYC trip data using:

- **Bronze** raw Parquet files in object storage
- **Silver** curated Iceberg table in the `silver` Polaris catalog
- **Gold** analytical Iceberg tables in the `gold` Polaris catalog
- **Spark / Trino / Jupyter** for exploration, transformation, and validation

We intentionally use a **three-level namespace** model to stay easy to understand and compatible with catalog patterns such as Unity Catalog:

- `silver.nyc_tlc.trips_unified`
- `gold.nyc_tlc.monthly_stats`
- `gold.nyc_tlc.zone_demand_daily`
- `gold.nyc_tlc.revenue_monthly`

For an automated mode with Apache Airflow, please check the [examples repository](https://github.com/OKDP/okdp-examples)

## Architecture overview

```text
                    +---------------------------+
                    |   External source files   |
                    |  NYC TLC monthly parquet  |
                    +-------------+-------------+
                                  |
                                  v
      +-------------------------------------------------------------+
      |                         BRONZE                              |
      |  Raw parquet in object storage                              |
      |                                                             |
      |  s3://bronze/mobility/nyc_tlc/yellow/                       |
      |  s3://bronze/mobility/nyc_tlc/green/                        |
      |                                                             |
      |  Jupyter notebooks: schema discovery, profiling, EDA,       |
      |  sampling, quality observations, Spark/Trino validation     |
      +---------------------------+---------------------------------+
                                  |
                                  | normalization, mapping,
                                  | standardization, lineage
                                  v
      +-------------------------------------------------------------+
      |                         SILVER                              |
      |  Polaris catalog: silver                                    |
      |  Storage root: s3://silver/                                 |
      |                                                             |
      |  Table: silver.nyc_tlc.trips_unified                        |
      |                                                             |
      |  Canonical trip schema built from yellow + green            |
      +---------------------------+---------------------------------+
                                  |
                                  | aggregations / business marts
                                  v
      +-------------------------------------------------------------+
      |                          GOLD                               |
      |  Polaris catalog: gold                                      |
      |  Storage root: s3://gold/                                   |
      |                                                             |
      |  Tables:                                                    |
      |   - gold.nyc_tlc.monthly_stats                              |
      |   - gold.nyc_tlc.zone_demand_daily                          |
      |   - gold.nyc_tlc.revenue_monthly                            |
      +-------------------------------------------------------------+
```


This project follows a [medallion](https://www.databricks.com/blog/what-is-medallion-architecture) architecture:

### 1. Bronze = raw exploration and understanding

The Bronze notebooks are **exploratory notebooks**. Their goal is not to publish final tables, but to:

- inspect schemas from raw Parquet files
- understand differences between datasets (`yellow`, `green`)
- profile nullability, data types, date fields, and business measures
- discover data quality rules and normalization rules
- validate access through Spark and Trino

At this stage, data remains close to the source structure.

### 2. Silver = canonical integrated table

The Silver layer uses what was learned in Bronze to build a **clean, unified, queryable business table**:

- harmonize column names between Yellow and Green trip datasets
- standardize timestamps and location fields
- add lineage columns such as `source_system`, `source_dataset`, `source_month`, `ingestion_ts`
- keep nullable fields where a measure exists in one source but not another
- publish the result as:
  - `silver.nyc_tlc.trips_unified`

This table is the **canonical detailed dataset** used by downstream analytics.

### 3. Gold = business-facing aggregates

Gold tables are derived from the Silver canonical table, not directly from Bronze.

They provide stable, analytics-ready outputs:

- `gold.nyc_tlc.monthly_stats`
- `gold.nyc_tlc.zone_demand_daily`
- `gold.nyc_tlc.revenue_monthly`

This separation keeps business metrics reproducible and consistent.


## Repository structure

```text
notebooks/
|-- _shared/
|   `-- utils.py
|-- bronze/
|   |-- index.ipynb
|   `-- mobility/
|       `-- nyc_trip/
|           |-- 01_pyspark_explore_yellow_tripdata.ipynb
|           |-- 02_trino_python_explore_yellow_tripdata.ipynb
|           `-- 03_trino_sql_explore_yellow_tripdata.ipynb
|-- silver/
|   `-- mobility/
|       |-- index.ipynb
|       `-- nyc_trip/
|           |-- 01_Issue-seaweedFS-build_trips.ipynb
|           `-- 01_build_trips.ipynb
`-- gold/
    `-- mobility/
        |-- index.ipynb
        `-- nyc_trip/
            `-- 01_build_monthly_stats.ipynb
```

Recommended intent by folder:

- `bronze/...`: exploration, profiling, raw dataset understanding
- `silver/...`: canonical model creation and validation
- `gold/...`: analytical marts and KPI tables
- `_shared/...`: common helpers reused across notebooks


## Catalogs and physical storage

### Bronze storage

- `s3://bronze/mobility/nyc_tlc/yellow/`
- `s3://bronze/mobility/nyc_tlc/green/`

Bronze stays outside Polaris because it is raw file-oriented storage.

### Silver catalog

- Catalog: `silver`
- Storage root: `s3://silver/`
- Main table: `silver.nyc_tlc.trips_unified`

### Gold catalog

- Catalog: `gold`
- Storage root: `s3://gold/`
- Main tables:
  - `gold.nyc_tlc.monthly_stats`
  - `gold.nyc_tlc.zone_demand_daily`
  - `gold.nyc_tlc.revenue_monthly`


## Recommended navigation path

Use the notebooks in this order:

1. Review the Bronze notebooks to understand the source data and schemas
2. Build the Silver trip table from Bronze raw data
3. Build Gold aggregates from the Silver table only

### Layer indexes

- [Bronze index](bronze/index.ipynb)
- [Silver mobility index](silver/mobility/index.ipynb)
- [Gold mobility index](gold/mobility/index.ipynb)


## Notebook areas

### Bronze

Bronze notebooks are used for exploration, source understanding, and schema inspection.

Current Bronze mobility notebooks:

- `bronze/mobility/nyc_trip/01_pyspark_explore_yellow_tripdata.ipynb`
- `bronze/mobility/nyc_trip/02_trino_python_explore_yellow_tripdata.ipynb`
- `bronze/mobility/nyc_trip/03_trino_sql_explore_yellow_tripdata.ipynb`

### Silver

Silver notebooks publish trusted conformed analytical datasets.

Current Silver mobility notebooks:

- `silver/mobility/nyc_trip/01_build_trips.ipynb`
- `silver/mobility/nyc_trip/01_Issue-seaweedFS-build_trips.ipynb`

### Gold

Gold notebooks publish consumer-facing aggregates and business-ready datasets.

Current Gold mobility notebooks:

- `gold/mobility/nyc_trip/01_build_monthly_stats.ipynb`


## Recommended execution path

### Standard path

For the standard showcase flow, use:

1. Bronze exploration notebooks
2. `silver/mobility/nyc_trip/01_build_trips.ipynb`
3. `gold/mobility/nyc_trip/01_build_monthly_stats.ipynb`

### Troubleshooting path

Use `silver/mobility/nyc_trip/01_Issue-seaweedFS-build_trips.ipynb` only when you need the environment-specific Silver notebook variant related to SeaweedFS write behavior.


## Shared runtime notes

- Spark, Polaris, Iceberg, and object-storage access are configured outside the notebooks
- Sensitive Spark settings are intentionally hidden from notebook code
- `_shared/utils.py` is reserved for reusable helpers when common logic needs to be centralized

## Current environment caveat

In the current environment, Iceberg writes may fail on SeaweedFS during multipart upload initialization.

The typical failure is `CreateMultipartUpload` with `403 Access Denied`.

This is a storage write-path issue and should not be confused with Silver or Gold modeling logic.


## Summary

This root notebook is the navigation and methodology entry point for the project.

In short:

- **Bronze** explores and documents the raw source data
- **Silver** creates one canonical Iceberg trip table
- **Gold** publishes curated analytical marts from Silver

Target published assets:

- `silver.nyc_tlc.trips_unified`
- `gold.nyc_tlc.monthly_stats`
